# Normalize Ingredients

This notebook parses and normalizes ingredient text into structured fields for downstream analysis.

In [1]:
import pandas as pd
import ast
from collections import Counter
import time
from ingredient_parser import parse_ingredient


def safe_parse_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []


csv_path = "../data/dataset_full_cleaned.csv"
df = pd.read_csv(csv_path, low_memory=False, nrows=10000)
df.drop(["Unnamed: 0"], axis=1, inplace=True)

df.head()

,title,ingredients,directions,link,source,NER,simplified_NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"['brown sugar', 'milk', 'vanilla', 'nuts', 'bu...","['brown sugar', 'milk', 'vanilla', 'nuts', 'bu..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"['beef', 'chicken breasts', 'cream of mushroom...","['beef', 'chicken breasts', 'cream of mushroom..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"['frozen corn', 'cream cheese', 'butter', 'gar...","['frozen corn', 'cream cheese', 'butter', 'gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"['chicken', 'chicken gravy', 'cream of mushroo...","['chicken', 'chicken gravy', 'cream of mushroo..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"['peanut butter', 'graham cracker crumbs', 'bu...","['peanut butter', 'graham cracker crumbs', 'bu..."


In [2]:
# Extract all ingredients across the full dataset (chunked, memory-safe)
ingredient_counter = Counter()
chunk_size = 100_000

for chunk in pd.read_csv(
    csv_path,
    nrows=10000,
    usecols=["ingredients"],
    chunksize=chunk_size,
    low_memory=False,
):
    parsed_lists = chunk["ingredients"].map(safe_parse_list)
    for ing_list in parsed_lists:
        cleaned = [str(item).strip() for item in ing_list if str(item).strip()]
        ingredient_counter.update(cleaned)

all_ingredients = sorted(ingredient_counter.keys(), key=str.lower)
print(f"Unique ingredients found: {len(all_ingredients):,}")
print(f"Total ingredient mentions: {sum(ingredient_counter.values()):,}")

Unique ingredients found: 28,554
Total ingredient mentions: 75,006


## Normalization Workflow

We build an ingredient vocabulary, inspect frequency, then parse each row into (item, quantity, unit) tuples.

In [3]:
unique = set()
chunk_size = 100_000

for chunk in pd.read_csv(
    csv_path,
    nrows=10000,
    usecols=["ingredients"],
    chunksize=chunk_size,
    low_memory=False,
):
    for val in chunk["ingredients"]:
        try:

            for raw in ast.literal_eval(str(val)):
                parsed = parse_ingredient(raw)
                item = parsed.name[0].text if parsed.name else None
                if item:
                    unique.add(item.strip().lower())
        except Exception:
            pass

items = sorted(unique)
print(f"Unique ingredients found: {len(items):,}")

Unique ingredients found: 7,305


In [4]:
# Quick inspection of extracted ingredient universe
ingredients_df = (
    pd.Series(ingredient_counter, name="count")
    .sort_values(ascending=False)
    .rename_axis("ingredient")
    .reset_index()
    .head(25)
)

display(ingredients_df)

,ingredient,count
0,1 tsp. vanilla,906
1,1 tsp. salt,882
2,1/2 tsp. salt,867
3,1 c. sugar,742
4,2 eggs,651
5,1 egg,484
6,2 c. sugar,480
7,1/4 tsp. salt,429
8,1/2 c. sugar,386
9,4 eggs,355


In [5]:
def normalize_ingredients(raw_ingredients):
    parsed_tuples = []
    for raw in raw_ingredients:
        parsed = parse_ingredient(raw)
        item = parsed.name[0].text if len(parsed.name) > 0 else None

        quantity = None
        unit = None

        if parsed.amount:
            amount_obj = parsed.amount[0]
            quantity_value = getattr(amount_obj, "quantity", None)
            if quantity_value is not None:
                try:
                    quantity = str(float(quantity_value))
                except ValueError:
                    quantity = str(quantity_value)

            unit_value = getattr(amount_obj, "unit", None)
            if unit_value is not None:
                unit = str(unit_value)

        parsed_tuples.append((item, quantity, unit))
    return parsed_tuples


demo_recipe = pd.read_csv(csv_path, nrows=10, low_memory=False).iloc[0]
raw_ingredients = ast.literal_eval(demo_recipe["ingredients"])
normalize_ingredients(raw_ingredients)

[('brown sugar', '1.0', 'cup'),
 ('evaporated milk', '0.5', 'cup'),
 ('vanilla', '0.5', 'teaspoon'),
 ('broken nuts', '0.5', 'cup'),
 ('butter', '2.0', 'Tbsps'),
 ('rice biscuits', '3.5', 'cup')]

In [6]:
# Apply normalize_ingredients row by row for easier debugging
print("Normalizing ingredients row by row...")

raw_ingredients_list = []
normalized_ingredients = []
t0 = time.time()
for idx, raw_value in enumerate(df["ingredients"]):
    try:
        raw_list = (
            ast.literal_eval(raw_value) if isinstance(raw_value, str) else raw_value
        )
        normalized_row = normalize_ingredients(raw_list)
        normalized_ingredients.append(normalized_row)
        only_items = list(map(lambda item: item[0], normalized_row))
        raw_ingredients_list.append(only_items)

        if idx < 5:
            print(f"Row {idx} OK -> {normalized_row[:3]}")
            print("Row ingredients (raw):", only_items[:3])
            print("\n")
    except Exception as e:
        normalized_ingredients.append([])
        raw_ingredients_list.append([])
        print(f"Row {idx} failed: {e}")

    if idx > 0 and idx % 1000 == 0:
        print(f"Processed {idx} rows in {time.time() - t0:.2f} seconds...")

df["normalized_ingredients"] = normalized_ingredients
df["raw_ingredients"] = raw_ingredients_list

print("Done.")
print(df[["title", "normalized_ingredients"]].head(3).to_string(index=False))
print(f"Total time taken: {time.time() - t0:.2f} seconds")

Normalizing ingredients row by row...
Row 0 OK -> [('brown sugar', '1.0', 'cup'), ('evaporated milk', '0.5', 'cup'), ('vanilla', '0.5', 'teaspoon')]
Row ingredients (raw): ['brown sugar', 'evaporated milk', 'vanilla']


Row 1 OK -> [('beef', '1.0', 'small jar'), ('chicken breasts', '4.0', ''), ('cream of mushroom soup', '1.0', 'can')]
Row ingredients (raw): ['beef', 'chicken breasts', 'cream of mushroom soup']


Row 2 OK -> [('pkg. frozen corn', '2.0', ''), ('pkg. cream cheese', '1.0', ''), ('butter', '0.3333333333333333', 'cup')]
Row ingredients (raw): ['pkg. frozen corn', 'pkg. cream cheese', 'butter']


Row 3 OK -> [('whole chicken', '1.0', ''), ('chicken gravy', '2.0', 'cans'), ('cream of mushroom soup', '1.0', 'can')]
Row ingredients (raw): ['whole chicken', 'chicken gravy', 'cream of mushroom soup']


Row 4 OK -> [('peanut butter', '1.0', 'cup'), ('graham cracker crumbs', '0.75', 'cup'), ('butter', '1.0', 'cup')]
Row ingredients (raw): ['peanut butter', 'graham cracker crumbs', '

In [7]:
df.head()

,title,ingredients,directions,link,source,NER,simplified_NER,normalized_ingredients,raw_ingredients
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"['brown sugar', 'milk', 'vanilla', 'nuts', 'bu...","['brown sugar', 'milk', 'vanilla', 'nuts', 'bu...","[(brown sugar, 1.0, cup), (evaporated milk, 0....","[brown sugar, evaporated milk, vanilla, broken..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"['beef', 'chicken breasts', 'cream of mushroom...","['beef', 'chicken breasts', 'cream of mushroom...","[(beef, 1.0, small jar), (chicken breasts, 4.0...","[beef, chicken breasts, cream of mushroom soup..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"['frozen corn', 'cream cheese', 'butter', 'gar...","['frozen corn', 'cream cheese', 'butter', 'gar...","[(pkg. frozen corn, 2.0, ), (pkg. cream cheese...","[pkg. frozen corn, pkg. cream cheese, butter, ..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"['chicken', 'chicken gravy', 'cream of mushroo...","['chicken', 'chicken gravy', 'cream of mushroo...","[(whole chicken, 1.0, ), (chicken gravy, 2.0, ...","[whole chicken, chicken gravy, cream of mushro..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"['peanut butter', 'graham cracker crumbs', 'bu...","['peanut butter', 'graham cracker crumbs', 'bu...","[(peanut butter, 1.0, cup), (graham cracker cr...","[peanut butter, graham cracker crumbs, butter,..."


## Export

Save the normalized dataset for modeling and retrieval experiments.

In [8]:
df.to_csv("../data/dataset_10000_normalized.csv", index=False)